![Piksel Sandbox](../../assets/piksel_header.png)

# Loading Data

## A. The dc.load function

Loading data is the operation that turns a query into pixels. The function `dc.load` takes the name of a product, an area on the ground, a time range, and a list of **measurements**, and returns the matching pixels as an `xarray.Dataset`.

Four inputs define every load:

- **product** — the dataset to read from, listed by `dc.list_products()`.
- **area** — a longitude range (`x`) and latitude range (`y`).
- **time** — a date or date range, for example `"2024"` or `("2024-01", "2024-06")`.
- **measurements** — the bands to return, listed by `dc.list_measurements()`.

Two further inputs control the output grid:

- **output_crs** — the coordinate reference system the result is projected into.
- **resolution** — the pixel size, given as `(y_step, x_step)` in the units of `output_crs`.

The same call shape works for every product in the datacube. Switching to a different product is a matter of changing `product=` and adjusting the measurement names.

## B. Outline

1. Connect to the datacube.
2. Define the area, time, and measurements to load.
3. Call `dc.load` and inspect the returned dataset.
4. Modify the query parameters and reload.

## C. Connecting to the datacube

The datacube connection is the same as in the previous notebook.

In [ ]:
import datacube

dc = datacube.Datacube(app="03_loading_data")

## D. Defining the area, time, and measurements

The area is given as longitude and latitude ranges. The example below covers a small region around **Lake Toba** in North Sumatra, roughly 0.1 degrees on each side. The time is a single year. The measurements are the four bands most often used for inspecting Sentinel-2 imagery: red, green, blue, and near-infrared.

The output grid is set explicitly. `s2_geomad_annual` has no default coordinate reference system or resolution, so both must be provided. EPSG:32647 is the local UTM zone, and 30 metre pixels keep the load tractable on the Sandbox.

In [ ]:
query = {
    "product": "s2_geomad_annual",
    "x": (98.80, 98.90),
    "y": (2.65, 2.55),
    "time": "2024",
    "measurements": ["red", "green", "blue", "nir"],
    "output_crs": "EPSG:32647",
    "resolution": (-30, 30),
}

## E. Loading the data

`dc.load` runs the query and returns an `xarray.Dataset`. Each measurement becomes a data variable; `time`, `y`, and `x` are the dimensions.

In [ ]:
ds = dc.load(**query)
ds

## F. Modifying the query parameters

The same call shape works for any product. The query below switches to `s2_l2a`, the per-scene Sentinel-2 surface reflectance product, narrows the time window to a single month, and requests only the red band. The grid parameters and area are unchanged.

The result is a different dataset with a different time density: `s2_l2a` returns one slice per overpass, where `s2_geomad_annual` returns one slice per year.

In [ ]:
query["product"] = "s2_l2a"
query["time"] = "2025-01"
query["measurements"] = ["red"]

ds = dc.load(**query)
ds

## G. Next steps

The next notebook describes the structure of the `xarray.Dataset` that `dc.load` returns: how to read its dimensions and coordinates, select subsets, and combine variables.

Continue to [04_xarray_for_odc.ipynb](04_xarray_for_odc.ipynb).